# Publikationsgrafiken und Quarto

Vollständiger synthetischer Ablauf ohne Excel-Server und ohne private Daten.
Installation im Repository: `python -m pip install -e ".[publication,dev]"`.
SVG/PNG benötigen Kaleido und Chrome; die Website zusätzlich Quarto CLI.
Die Beispiele sind keine wissenschaftlichen Referenzwerte oder Simulationsergebnisse.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "pyproject.toml").exists(), "Im Repository oder notebooks/ starten"
sys.path.insert(0, str(root / "scripts"))
from publication_example import example_project, example_figures, provenance, build, mobility_figure
from pexl import reporting
from pexl.plot import plotly
from pexl.plot.export import export_figure
from IPython.display import display, HTML, SVG, Image
output = root / "notebooks/local_exports/publication"


## 1. Projekt → Bericht → Figur

Dieselbe Pipeline wie im Streamlit-Dashboard. Für echte Daten ersetzt
`Project.from_excel("projektexport.xlsx")` das synthetische Projekt.
Die Berechnung der Arbeitsmappe muss bereits abgeschlossen sein.

In [ ]:
project = example_project()
report = reporting.materialize(project.view, "primary_energy_balance")
figure = plotly.render(report)
display(report["frame"])
display(HTML(figure.to_html(include_plotlyjs=True, full_html=False)))

## 2. Interaktives HTML und druckfähige SVG/PNG

Die Herkunft bezeichnet die Quelldaten, nicht die Exportzeit. Für echte Projekte
Quelle, Szenarien, Modellversion und Berechnungsdatum ersetzen. Die Schemaversion
wird separat angegeben. Fehlende Berechnungsdaten ausdrücklich als unbekannt benennen.
Die Originalfigur bleibt unverändert; alle Exporte erhalten dieselbe sichtbare Fußnote.

In [ ]:
figures = example_figures()
for extension in ("html", "svg", "png"):
    export_figure(figure, output / f"primary_energy_balance.{extension}",
                  provenance=provenance(figures["primary_energy_balance"][1]))
display(SVG(filename=str(output / "primary_energy_balance.svg")))

## 3. THG, Stromflüsse und Methoden

Der gemeinsame Beispielworkflow exportiert auch THG-Detaildiagramme,
Stromflüsse mit Bezug positiv/Einspeisung negativ und die Dichtekurve.
Dichte verwendet `Zielwert.ZQSynergy()` mit unveränderten historischen Parametern.
Der Kontextfaktor ist kein realer Energiefluss.

Die THG-Grafiken übernehmen die bestehenden L1/L2/L3-Bänder und kumulierten Werte.
50 Jahre (2025–2075), nationales Zieljahr und Dekarbonisierungspfad sind verschiedene
Angaben. Zieljahr und jährlicher Pfad fehlen im synthetischen Datensatz und werden
nicht erfunden. Siehe `docs/development/publication.md` für Herleitung und Grenzen.

In [ ]:
paths = build(output)
assert len(paths) == 15
assert all(path.exists() and path.stat().st_size > 0 for path in paths)
display(Image(filename=str(output / "electricity.png")))
display(SVG(filename=str(output / "density.svg")))
display(SVG(filename=str(output / "gwp_ee_detail.svg")))

## 4. Nutzung und Flächenanteile im Mobilitätsprototyp auswählen

Die folgenden Budgetwerte sind ausschließlich synthetisch. Für eine nationale
Zuordnung müssen die geprüften Werte aus `cfm_budget_*` der konkreten Modellversion
übernommen werden. Einheit: kWhPE/(m² NGF a). Flächenanteile müssen zusammen 1 ergeben.
Der gemeinsame Beispielcode gewichtet wie die vorhandene Schemaformel und trennt
Budget, Bedarf und den positiven/negativen Bilanzbeitrag. Mobilität ist eingeschlossen.

In [ ]:
shares = {"Wohnen": 0.7, "Büro": 0.3}
budgets = {"Wohnen": 20.0, "Büro": 10.0}  # synthetisch, keine nationalen Referenzen
mobility = mobility_figure(shares, budgets, actual=22.0)
display(HTML(mobility.to_html(include_plotlyjs=True, full_html=False)))

## 5. Quarto-Website lokal bauen

Die Website ruft denselben Beispielworkflow als Pre-render-Schritt auf. HTML wird
per iframe eingebunden, SVG und PNG sind separat verlinkt. Quarto ist ein externes
Build-Werkzeug; falls es fehlt, bleiben die erzeugten Grafiken direkt nutzbar.
Dieser Schritt veröffentlicht nichts. Die Streamlit-App läuft weiterhin separat
mit `python -m streamlit run apps/dashboard/main.py`.

In [ ]:
import shutil
import subprocess
quarto = shutil.which("quarto")
if quarto:
    subprocess.run([quarto, "render", "website"], cwd=root, check=True)
    print(root / "website/_site/index.html")
else:
    print("Quarto CLI fehlt: nach Installation 'quarto render website' im Repository ausführen.")